In [0]:
# notebook 2 - camada SILVER
# le a bronze, limpa, coloca os nomes em portugues e grava as 7 tabelas silver com o nome que a atividade pede
# a gold le exatamente esses nomes: tb_info_filmes, tb_financeiro_filmes, tb_metricas_engajamento,
# tb_avaliacoes_usuarios, tb_generos, tb_pessoas_empresas e tb_cotacao_dolar
import importlib, sys
if 'medalhao_config' in sys.modules:
    del sys.modules['medalhao_config']
from medalhao_config import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from check import dq_check, dq_check_unique, dq_results
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
print(bronze_schema, silver_schema)
print(dq_check, dq_check_unique,dq_results)


medalhao_cinema.bronze medalhao_cinema.silver
<function dq_check at 0xffe611b380e0> <function dq_check_unique at 0xffe611b38040> [Row(table_name='silver.tb_info_filmes', check_name='id_filme não nulo', total_rows=97879, failed_rows=0, passed=True, checked_at=datetime.datetime(2026, 10, 16, 17, 23, 38, 476688)), Row(table_name='silver.tb_info_filmes', check_name='id_filme único', total_rows=97879, failed_rows=0, passed=True, checked_at=datetime.datetime(2026, 10, 16, 17, 23, 43, 419085)), Row(table_name='silver.tb_cotacao_dolar', check_name='1 linha por dia', total_rows=46, failed_rows=0, passed=True, checked_at=datetime.datetime(2026, 10, 16, 17, 24, 11, 272609)), Row(table_name='silver.tb_cotacao_dolar', check_name='cotacao preenchida', total_rows=46, failed_rows=0, passed=True, checked_at=datetime.datetime(2026, 10, 16, 17, 24, 15, 393005)), Row(table_name='silver.tb_info_filmes', check_name='id_filme não nulo', total_rows=97879, failed_rows=0, passed=True, checked_at=datetime.dateti

In [0]:
# funcoes que repito em todas as tabelas
# mais_recente: quando o mesmo id vem 2x na bronze (append), fico so com a linha da ingestao mais nova
# numero_seguro: cast que devolve NULL em vez de quebrar quando vem texto (ANSI do databricks quebra no cast normal)
def mais_recente(df, chave):
    w = Window.partitionBy(chave).orderBy(F.col("ingestion_datetime").desc())
    return df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

def votos_inteiros(coluna):
    texto = F.trim(F.col(coluna).cast("string"))

    # Aceita inteiro ou representação decimal com fração zero.
    formato_valido = texto.rlike(
        r"^\+?\d+(?:[.,]0+)?$"
    )

    # Retira somente a parte decimal composta por zeros.
    parte_inteira = F.regexp_replace(
        texto,
        r"[.,]0+$",
        ""
    )

    # Limita o cast a uma representação reconhecida.
    inteiro_texto = F.when(
        formato_valido,
        parte_inteira
    )

    return inteiro_texto

def salvar_silver(df, nome):
    # Na Bronze, o nome obrigatório continua ingestion_datetime.
    # Na Silver, todos os nomes devem estar em português.
    if "ingestion_datetime" in df.columns:
        df = df.withColumnRenamed(
            "ingestion_datetime",
            "data_hora_ingestao"
        )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table(silver_schema, nome))
    )

    print(f"Gravado: {table(silver_schema, nome)}")

## silver.tb_info_filmes

In [0]:
def numero_seguro(coluna, tipo):
    return F.expr(f"try_cast({coluna} as {tipo})")
numero_seguro("orcamento_txt", "decimal(18,2)")

Column<'try_cast(orcamento_txt as decimal(18,2))'>

In [0]:
df_silver_filmes = (
    df_silver_filmes
    .withColumn(
        "duracao_minutos",
        numero_seguro("duracao_minutos", "int")
    )
    .select(
        "id_filme",
        "titulo",
        "titulo_original",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme",
        "sinopse",
        "frase_divulgacao",
        "ingestion_datetime"
    )
)

In [0]:
# pegando como referencia o uso de regex que foi falado na aula pratica, junto da parte que uso como base o notebook 
# consertado: nome da tabela (tb_info_filmes), dedup pela ingestao mais recente, data com try (nao quebra no ANSI)
# e o status normalizado uma vez so (tudo que nao e letra vira espaco) antes de traduzir
# vamos fazer o tratamento multidado-formato pois tem muitas datas em varios formatos 
df_bronze_filmes=spark.table(table(bronze_schema,"tb_movies_info"))
                             
df_silver_filmes=(df_bronze_filmes
    .withColumnRenamed("id","id_filme")
    .withColumnRenamed("title","titulo")
    .withColumnRenamed("original_title","titulo_original")
    .withColumnRenamed("release_date","data_lancamento")
    .withColumnRenamed("runtime","duracao_minutos")
    .withColumnRenamed("original_language","idioma_original")
    .withColumnRenamed("status","status_filme")
    .withColumnRenamed("overview","sinopse")
    .withColumnRenamed("tagline","frase_divulgacao")
)

df_silver_filmes=df_silver_filmes.withColumn("id_filme",F.trim(F.col("id_filme").cast("string")))
df_silver_filmes=mais_recente(df_silver_filmes.filter(F.col("id_filme").isNotNull()),"id_filme")
# pegando da aula onde foi abordado o uso de  regex, para limpar 
status_limpo=F.trim(F.regexp_replace(F.lower(F.col("status_filme")),"[^a-z]+"," "))

# 
df_silver_filmes = df_silver_filmes.withColumn(
    "status_filme",
    F.when(status_limpo == "released", "Lançado")
     .when(status_limpo == "post production", "Pós-Produção")
     .when(status_limpo == "in production", "Em Produção")
     .when(status_limpo == "planned", "Planejado")
     .when(status_limpo == "rumored", "Rumores")
     .when(status_limpo == "canceled", "Cancelado")
     .otherwise("Não Informado")
)
# conserta o poblema de multiformato usammos o colescence para evitar a duplicaco
data_txt = F.trim(
    F.col("data_lancamento").cast("string")
)

# Política para datas ambíguas:
# dia/mês tem prioridade sobre mês/dia.
# Exemplo: 01/02/2020 será interpretado como 1º de fevereiro.
formatos_data = [
    "yyyy-MM-dd",
    "dd/MM/yyyy",
    "MM/dd/yyyy",
    "yyyy/MM/dd",
    "dd-MM-yyyy",
    "MM-dd-yyyy",
]

df_silver_filmes = (
    df_silver_filmes
    .withColumn(
        "data_lancamento",
        F.coalesce(
            *[
                F.to_date(
                    F.try_to_timestamp(
                        data_txt,
                        F.lit(formato)
                    )
                )
                for formato in formatos_data
            ]
        )
    )
    .withColumn(
        "ano_lancamento",
        F.year("data_lancamento")
    )
)
dq_check("silver.tb_info_filmes","id_filme não nulo",df_silver_filmes,F.col("id_filme").isNotNull())
dq_check_unique("silver.tb_info_filmes","id_filme único",df_silver_filmes,["id_filme"])

# cria o schema e salva com o nome pedido na atividade
salvar_silver(df_silver_filmes,"tb_info_filmes")
display(df_silver_filmes.limit(100))

[PASS] silver.tb_info_filmes | id_filme não nulo | 0/97879 linhas falharam
[PASS] silver.tb_info_filmes | id_filme único | 0 chaves duplicadas de 97879 linhas
Gravado: medalhao_cinema.silver.tb_info_filmes


id_filme,tconst,titulo,titulo_original,idioma_original,data_lancamento,duracao_minutos,status_filme,sinopse,frase_divulgacao,ingestion_datetime,ano_lancamento
1000075,tt23049322,Largo Winch III : Le prix de l'argent,Largo Winch III : Le prix de l'argent,fr,2024-10-16,0,Em Produção,null,null,2026-10-16T05:01:44.984Z,2024
1000146,tt19408362,BARRACO DE FAMÍLIA,Barraco de Família,pt,2023-04-19,85,Lançado,"Cleide is visited by her daughter Kellen, a successful funk artist who, after a year without any news, reappears in the suburban village where the family lives.",null,2026-10-16T05:01:44.984Z,2023
1000194,tt21281774,BANK ROBBERS: THE LAST GREAT HEIST,Los ladrones: la verdadera historia del robo del siglo,es,2022-08-10,109,Lançado,"Through candid interviews, the perpetrators of Argentina's most notorious bank heist detail how — and why — they carried out the radical 2006 operation.",null,2026-10-16T05:01:44.984Z,2022
1000210,tt7721078,Techno,Techno,pl,2018-06-19,24,Lançado,null,null,2026-10-16T05:01:44.984Z,2018
1000409,tt21230186,Strangers by Night,Une nuit,fr,2023-07-05,90,Lançado,\In a crowded subway train,"a woman pushes a man. They argue. Their little joust """"\""""barks loudly""""\""""""""",2026-10-16T05:01:44.984Z,2023
1000458,tt21074700,C600: The Singularity,C600: The Singularity,en,2022-06-17,15,Lançado,C600 Chapter 4 is on a whole new level of this epic sci-fi series. We see what happens to the agents from the events at the end of Chapter 2. (While Chapter 4 was a side story of Mr. X). The agents and Mr. X realises that they must become allies to win this fight.,null,2026-10-16T05:01:44.984Z,2022
1000563,tt16148552,The Lost Boys,Le paradis,fr,2023-03-23,83,Lançado,"In a youth correctional facility, Joe is preparing his return to society. But William’s arrival turns his desire for freedom into desire of another kind. Behind fences and cell walls, passions begin to play havoc with the need for liberty.",null,2026-10-16T05:01:44.984Z,2023
1000593,tt7603760,Taniel,Taniel,en,2018-03-29,20,Lançado,"\""""\""""Taniel""""\"""" looks at the last months of poet Taniel Varoujan's life""""/e4w0H7RKwwYz5hzQYnmQqesWOfo.jpg",null,2026-10-16T05:01:44.984Z,2018
1000618,tt21378332,To The North,Spre nord,ro,2022-08-31,122,Lançado,null,null,2026-10-16T05:01:44.984Z,2022
1000622,tt16464122,Elvis and the USS Arizona,Elvis and the USS Arizona,en,2021-11-14,58,Lançado,The surprising story of how one of music's biggest icons helped to establish a USS Arizona memorial in Pearl Harbor. Elvis’ fundraising concert drew public attention to the plight and helped to galvanize efforts to finish the USS Arizona Memorial as it stands today.,The story of how one of music's biggest icons helped to establish a USS Arizona memorial.,2026-10-16T05:01:44.984Z,2021


## silver.tb_cotacao_dolar

In [0]:
# faço a cotacao antes do financeiro porque o financeiro precisa dela pra calcular o BRL
# a API nao tem fim de semana/feriado: crio todos os dias do periodo e preencho o buraco com o ultimo dia util (forward fill)
df_bronze_cotacao=spark.table(table(bronze_schema,"tb_cotacao_dolar"))

w_dia=Window.partitionBy("data_cotacao").orderBy(F.col("data_hora_cotacao").desc(),F.col("ingestion_datetime").desc())
cotacao_dia=(df_bronze_cotacao
    .withColumn("data_cotacao",F.to_date("data_hora_cotacao"))
    .withColumn("rn",F.row_number().over(w_dia))
    .filter("rn = 1")
    .select("data_cotacao","cotacao_compra","cotacao_venda"))

limites=cotacao_dia.agg(F.min("data_cotacao").alias("ini"),F.max("data_cotacao").alias("fim"))
calendario=limites.select(F.explode(F.sequence("ini",F.greatest("fim",F.current_date()))).alias("data_cotacao"))

w_ffill=Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding,0)
df_silver_cotacao=(calendario
    .join(cotacao_dia,"data_cotacao","left")
    .withColumn("dia_preenchido",F.col("cotacao_compra").isNull())
    .withColumn("cotacao_compra",F.last("cotacao_compra",ignorenulls=True).over(w_ffill).cast("decimal(10,4)"))
    .withColumn("cotacao_venda",F.last("cotacao_venda",ignorenulls=True).over(w_ffill).cast("decimal(10,4)")))

dq_check_unique("silver.tb_cotacao_dolar","1 linha por dia",df_silver_cotacao,["data_cotacao"])
dq_check("silver.tb_cotacao_dolar","cotacao preenchida",df_silver_cotacao,F.col("cotacao_compra").isNotNull())

salvar_silver(df_silver_cotacao,"tb_cotacao_dolar")
display(df_silver_cotacao.orderBy(F.col("data_cotacao").desc()))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] silver.tb_cotacao_dolar | 1 linha por dia | 0 chaves duplicadas de 46 linhas
[PASS] silver.tb_cotacao_dolar | cotacao preenchida | 0/46 linhas falharam
Gravado: medalhao_cinema.silver.tb_cotacao_dolar


data_cotacao,cotacao_compra,cotacao_venda,dia_preenchido
2026-10-16,5.0515,5.0521,true
2026-10-15,5.0515,5.0521,false
2026-10-14,5.0435,5.0441,false
2026-10-13,5.0090,5.0096,false
2026-10-12,4.9886,4.9892,true
2026-10-11,4.9886,4.9892,true
2026-10-10,4.9886,4.9892,true
2026-10-09,4.9886,4.9892,false
2026-10-08,5.0113,5.0119,false
2026-10-07,4.9929,4.9935,false


## silver.tb_financeiro_filmes

In [0]:
# limpa orcamento/receita: texto de ausencia vira NULL, tira simbolo de moeda e ponto/virgula de milhar, <= 0 vira NULL
# BRL = USD x cotacao de compra mais recente da silver.tb_cotacao_dolar
# lucro so e calculado quando tem orcamento e receita (senao fica NULL, nao inventa zero); margem so quando receita > 0
def limpar_valor(coluna):
    txt = F.upper(F.trim(F.col(coluna).cast("string")))

    # Ausências explícitas.
    txt = F.when(
        txt.isin(
            "", "UNKNOWN", "NÃO INFORMADO", "NAO INFORMADO",
            "N/A", "NA", "NULL", "NONE", "NAN", "-"
        ),
        F.lit(None).cast("string")
    ).otherwise(txt)

    # Remove identificadores de dólar no início ou no fim.
    txt = F.trim(
        F.regexp_replace(txt, r"^(?:USD|US\$|\$)\s*", "")
    )
    txt = F.trim(
        F.regexp_replace(txt, r"\s*(?:USD|US\$|\$)$", "")
    )

    # Valida o formato completo antes de remover separadores.
    return (
        # Inteiro: 12345
        F.when(txt.rlike(r"^-?\d+$"), txt)

        # Decimal sem milhar: 123,45 ou 123.45
        .when(
            txt.rlike(r"^-?\d+[.,]\d{1,2}$"),
            F.regexp_replace(txt, ",", ".")
        )

        # Brasileiro com centavos: 1.234.567,89
        .when(
            txt.rlike(r"^-?\d{1,3}(?:\.\d{3})+,\d{1,2}$"),
            F.regexp_replace(
                F.regexp_replace(txt, r"\.", ""), ",", "."
            )
        )

        # Americano com centavos: 1,234,567.89
        .when(
            txt.rlike(r"^-?\d{1,3}(?:,\d{3})+\.\d{1,2}$"),
            F.regexp_replace(txt, ",", "")
        )

        # Milhar sem centavos: 1.000 ou 1.000.000
        .when(
            txt.rlike(r"^-?\d{1,3}(?:\.\d{3})+$"),
            F.regexp_replace(txt, r"\.", "")
        )

        # Milhar sem centavos: 1,000 ou 1,000,000
        .when(
            txt.rlike(r"^-?\d{1,3}(?:,\d{3})+$"),
            F.regexp_replace(txt, ",", "")
        )

        # Milhões: 34.0M, 34,0M ou 34 M
        # O limite de dígitos protege o cast intermediário.
        .when(
            txt.rlike(r"^-?\d{1,12}(?:[.,]\d{1,2})?\s*M$"),
            (
                F.regexp_replace(
                    F.regexp_replace(txt, r"\s*M$", ""), ",", "."
                ).cast("decimal(20,2)") * F.lit(1000000)
            ).cast("string")
        )

        .otherwise(F.lit(None).cast("string"))
    )


# Cotação de compra mais recente até a data de execução.
# Se a última cotação estiver inválida, interrompe antes de salvar.
linha_cotacao = (
    spark.table(table(silver_schema, "tb_cotacao_dolar"))
    .filter(F.col("data_cotacao") <= F.current_date())
    .orderBy(F.col("data_cotacao").desc())
    .select("data_cotacao", "cotacao_compra")
    .first()
)

if (
    linha_cotacao is None
    or linha_cotacao["cotacao_compra"] is None
    or linha_cotacao["cotacao_compra"] <= 0
):
    raise ValueError(
        "Cotação mais recente ausente ou inválida. "
        "Execute e verifique a criação de silver.tb_cotacao_dolar."
    )

cotacao = linha_cotacao["cotacao_compra"]

print(
    "Data da cotação:", linha_cotacao["data_cotacao"],
    "| Cotação de compra usada:", cotacao
)


# Leitura da Bronze: nenhuma alteração na tabela de origem.
df_bronze_fin = spark.table(
    table(bronze_schema, "tb_movies_financials")
)

df_silver_fin = (
    df_bronze_fin
    .withColumn("id_filme", F.trim(F.col("id").cast("string")))
    .filter(
        F.col("id_filme").isNotNull()
        & (F.col("id_filme") != "")
    )
)

# Mantém a ingestão mais recente de cada filme.
df_silver_fin = mais_recente(df_silver_fin, "id_filme")


# Limpeza e conversão segura para decimal.
df_silver_fin = (
    df_silver_fin
    .withColumn("orcamento_txt", limpar_valor("budget"))
    .withColumn("receita_txt", limpar_valor("revenue"))

    .withColumn(
        "orcamento_usd",
        numero_seguro("orcamento_txt", "decimal(18,2)")
    )
    .withColumn(
        "receita_usd",
        numero_seguro("receita_txt", "decimal(18,2)")
    )

    # Zero e negativos representam ausência.
    .withColumn(
        "orcamento_usd",
        F.when(F.col("orcamento_usd") > 0, F.col("orcamento_usd"))
    )
    .withColumn(
        "receita_usd",
        F.when(F.col("receita_usd") > 0, F.col("receita_usd"))
    )

    .withColumn(
        "cotacao_usada",
        F.lit(cotacao).cast("decimal(10,4)")
    )

    # Conversão de USD para BRL.
    .withColumn(
        "orcamento_brl",
        F.round(
            F.col("orcamento_usd") * F.col("cotacao_usada"), 2
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "receita_brl",
        F.round(
            F.col("receita_usd") * F.col("cotacao_usada"), 2
        ).cast("decimal(18,2)")
    )

    # NULL é preservado se faltar orçamento ou receita.
    # Lucro negativo é válido: representa prejuízo.
    .withColumn(
        "lucro_usd",
        (F.col("receita_usd") - F.col("orcamento_usd"))
        .cast("decimal(18,2)")
    )
    .withColumn(
        "lucro_brl",
        (F.col("receita_brl") - F.col("orcamento_brl"))
        .cast("decimal(18,2)")
    )

    # Margem = lucro / receita × 100.
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            (F.col("receita_usd") > 0)
            & F.col("orcamento_usd").isNotNull(),
            F.round(
                (F.col("lucro_usd") / F.col("receita_usd")) * 100,
                2
            )
        ).cast("decimal(10,2)")
    )

.select(
        "id_filme",
        "orcamento_usd",
        "receita_usd",
        "orcamento_brl",
        "receita_brl",
        "lucro_usd",
        "lucro_brl",
        "margem_lucro_percentual",
        "cotacao_usada",

        # Informação adicional para conferir a conversão.
        F.lit(linha_cotacao["data_cotacao"])
         .cast("date")
         .alias("data_cotacao_usada"),

        # A Bronze mantém ingestion_datetime.
        # Na Silver, o nome fica em português.
        F.col("ingestion_datetime")
         .alias("data_hora_ingestao")
    )

)


# Verificações de qualidade.
dq_check_unique(
    "silver.tb_financeiro_filmes",
    "id_filme único",
    df_silver_fin,
    ["id_filme"]
)

dq_check(
    "silver.tb_financeiro_filmes",
    "orcamento > 0 ou NULL",
    df_silver_fin,
    F.col("orcamento_usd").isNull()
    | (F.col("orcamento_usd") > 0)
)

dq_check(
    "silver.tb_financeiro_filmes",
    "receita > 0 ou NULL",
    df_silver_fin,
    F.col("receita_usd").isNull()
    | (F.col("receita_usd") > 0)
)


# Grava somente a Silver.
salvar_silver(
    df_silver_fin,
    "tb_financeiro_filmes"
)

financeiro_gravado = spark.table(
    table(silver_schema, "tb_financeiro_filmes")
)

financeiro_gravado.printSchema()

display(
    financeiro_gravado.select(
        "id_filme",
        "orcamento_usd",
        "receita_usd",
        "cotacao_usada",
        "data_cotacao_usada",
        "orcamento_brl",
        "receita_brl",
        "lucro_usd",
        "lucro_brl",
        "margem_lucro_percentual",
        "data_hora_ingestao"
    ).limit(20)
)

Data da cotação: 2026-10-16 | Cotação de compra usada: 5.0515
[PASS] silver.tb_financeiro_filmes | id_filme único | 0 chaves duplicadas de 99006 linhas
[PASS] silver.tb_financeiro_filmes | orcamento > 0 ou NULL | 0/99006 linhas falharam
[PASS] silver.tb_financeiro_filmes | receita > 0 ou NULL | 0/99006 linhas falharam
Gravado: medalhao_cinema.silver.tb_financeiro_filmes
root
 |-- id_filme: string (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- margem_lucro_percentual: decimal(10,2) (nullable = true)
 |-- cotacao_usada: decimal(10,4) (nullable = true)
 |-- data_cotacao_usada: date (nullable = true)
 |-- data_hora_ingestao: timestamp (nullable = true)



id_filme,orcamento_usd,receita_usd,cotacao_usada,data_cotacao_usada,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual,data_hora_ingestao
1000004,null,null,5.0515,2026-10-16,null,null,null,null,null,2026-10-16T05:01:52.049Z
1000005,null,null,5.0515,2026-10-16,null,null,null,null,null,2026-10-16T05:01:52.049Z
1000007,null,null,5.0515,2026-10-16,null,null,null,null,null,2026-10-16T05:01:52.049Z
1000011,null,null,5.0515,2026-10-16,null,null,null,null,null,2026-10-16T05:01:52.049Z
1000014,null,null,5.0515,2026-10-16,null,null,null,null,null,2026-10-16T05:01:52.049Z
1000030,null,null,5.0515,2026-10-16,null,null,null,null,null,2026-10-16T05:01:52.049Z
1000054,null,null,5.0515,2026-10-16,null,null,null,null,null,2026-10-16T05:01:52.049Z
1000058,4700000.00,null,5.0515,2026-10-16,23742050.00,null,null,null,null,2026-10-16T05:01:52.049Z
1000059,null,null,5.0515,2026-10-16,null,null,null,null,null,2026-10-16T05:01:52.049Z
1000073,6000000.00,null,5.0515,2026-10-16,30309000.00,null,null,null,null,2026-10-16T05:01:52.049Z


## silver.tb_metricas_engajamento

In [0]:
# popularidade vem com ponto e virgula misturados: acerto o separador decimal antes do cast
# column shift: texto nas colunas de nota/voto vira NULL pelo try_cast (nao quebra o pipeline)
# nota fora de 0-10 (ex.: 78 por erro de escala) vira NULL; voto/popularidade negativo vira NULL
pop_original=F.trim(F.col("popularity").cast("string")) # problema 1 fix para a entrega 3 ! questao 2 esta gerendo uma p(x;y)... resolverr

formato_simples = pop_original.rlike(
    r"^[+-]?\d+(?:[.,]\d+)?$"
)
formato_br = pop_original.rlike(
    r"^[+-]?\d{1,3}(?:\.\d{3})+,\d+$"
)

# Milhar e decimal: 1,234.56
formato_us = pop_original.rlike(
    r"^[+-]?\d{1,3}(?:,\d{3})+\.\d+$"
)


pop = (
    F.when(
        formato_br,
        F.regexp_replace(
            F.regexp_replace(pop_original, r"\.", ""),
            ",",
            "."
        )
    )
    .when(
        formato_us,
        F.regexp_replace(pop_original, ",", "")
    )
    .when(
        formato_simples,
        F.regexp_replace(pop_original, ",", ".")
    )
)                                 

def nota_txt(coluna):
    return F.regexp_replace(F.trim(F.col(coluna).cast("string")),",",".")

df_bronze_met=spark.table(table(bronze_schema,"tb_movies_metrics"))

df_silver_met=(df_bronze_met
    .withColumn("id_filme",F.trim(F.col("id").cast("string")))
    .filter(F.col("id_filme").isNotNull())
    .withColumn("pop_txt",pop)
    .withColumn("tmdb_txt",nota_txt("vote_average"))
    .withColumn("imdb_txt",nota_txt("averageRating"))
    .withColumn("votos_tmdb_txt",nota_txt("vote_count"))
    .withColumn("votos_imdb_txt",nota_txt("numVotes")))
df_silver_met=mais_recente(df_silver_met,"id_filme")

# comsertando o problema de cofnundir data com popularidade !!! erro da atividade 2 que coment
# Registros identificados no CSV com conteúdo incompatível
# entre as colunas de métricas.
# A popularidade original não pôde ser recuperada.
# Corrige somente os pares ID/valor inspecionados, preservando
# uma eventual atualização futura com outro valor. 

popularidade_inconsistente = (
    ((F.col("id_filme") == "809905") & (F.col("pop_txt") == "2020"))
    | ((F.col("id_filme") == "658829") & (F.col("pop_txt") == "2019"))
    | ((F.col("id_filme") == "557809") & (F.col("pop_txt") == "2018"))
    | (
    (F.col("id_filme") == "418746")
    & (F.col("pop_txt") == "1969")
    & (F.upper(F.trim(F.col("averageRating"))) == "ITALIAN")
) # colocar os anos corretos pq ja esta sendo a 4 vez que volto aqui. 
)

df_silver_met = df_silver_met.withColumn(
    "pop_txt",
    F.when(
        popularidade_inconsistente,
        F.lit(None).cast("string")
    ).otherwise(F.col("pop_txt"))
)

df_silver_met=(df_silver_met
    .withColumn("popularidade",numero_seguro("pop_txt","double"))
    .withColumn("nota_media_tmdb",numero_seguro("tmdb_txt","double"))
    .withColumn("nota_media_imdb",numero_seguro("imdb_txt","double"))
    .withColumn(
    "votos_tmdb_inteiro_txt",
    votos_inteiros("votos_tmdb_txt")
)
.withColumn(
    "votos_imdb_inteiro_txt",
    votos_inteiros("votos_imdb_txt")
)
.withColumn(
    "qtd_votos_tmdb",
    numero_seguro("votos_tmdb_inteiro_txt", "int")
)
.withColumn(
    "qtd_votos_imdb",
    numero_seguro("votos_imdb_inteiro_txt", "int")
)
    .withColumn("popularidade",F.when(F.col("popularidade")>=0,F.col("popularidade")))
    .withColumn("nota_media_tmdb",F.when(F.col("nota_media_tmdb").between(0,10),F.col("nota_media_tmdb")))
    .withColumn("nota_media_imdb",F.when(F.col("nota_media_imdb").between(0,10),F.col("nota_media_imdb")))
    .withColumn("qtd_votos_tmdb",F.when(F.col("qtd_votos_tmdb")>=0,F.col("qtd_votos_tmdb")))
    .withColumn("qtd_votos_imdb",F.when(F.col("qtd_votos_imdb")>=0,F.col("qtd_votos_imdb")))
    .select("id_filme","popularidade","nota_media_tmdb","qtd_votos_tmdb","nota_media_imdb","qtd_votos_imdb","ingestion_datetime"))

dq_check_unique("silver.tb_metricas_engajamento","id_filme único",df_silver_met,["id_filme"])
dq_check("silver.tb_metricas_engajamento","nota tmdb 0-10 ou NULL",df_silver_met,
         F.col("nota_media_tmdb").isNull()|F.col("nota_media_tmdb").between(0,10))

salvar_silver(df_silver_met,"tb_metricas_engajamento")
display(df_silver_met.limit(10))

[PASS] silver.tb_metricas_engajamento | id_filme único | 0 chaves duplicadas de 99013 linhas
[PASS] silver.tb_metricas_engajamento | nota tmdb 0-10 ou NULL | 0/99013 linhas falharam
Gravado: medalhao_cinema.silver.tb_metricas_engajamento


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb,ingestion_datetime
1000007,0.6,0.0,0,4.7,10,2026-10-16T05:01:55.719Z
1000073,13.212,6.8,15,null,236,2026-10-16T05:01:55.719Z
1000079,1.4,0.0,0,5.7,342,2026-10-16T05:01:55.719Z
1000096,0.71,10.0,1,6.3,695,2026-10-16T05:01:55.719Z
1000099,0.6,0.0,0,8.6,33,2026-10-16T05:01:55.719Z
1000110,1.38,0.0,0,6.9,263,2026-10-16T05:01:55.719Z
1000116,0.692,7.0,1,6.7,17,2026-10-16T05:01:55.719Z
1000130,1.751,5.778,9,6.3,843,2026-10-16T05:01:55.719Z
1000173,null,null,0,0.0,null,2026-10-16T05:01:55.719Z
1000237,5.452,6.7,18,null,1023,2026-10-16T05:01:55.719Z


In [0]:
display(
    spark.table(table(silver_schema, "tb_metricas_engajamento"))
    .filter(F.col("id_filme").isin("809905", "658829", "557809", "565770"))
    .select("id_filme", "popularidade", "nota_media_imdb", "qtd_votos_imdb")
)

id_filme,popularidade,nota_media_imdb,qtd_votos_imdb
557809,null,null,null
565770,2994.357,5.9,119501
658829,null,null,null
809905,null,null,null


## silver.tb_avaliacoes_usuarios

In [0]:
# nota fora de 0-10 vira NULL; comentario vazio ou so espaco vira "Sem comentário"
# depois de limpar tiro as avaliacoes 100% iguais (filme + usuario + nota + comentario)
df_bronze_rev=spark.table(table(bronze_schema,"tb_movies_reviews"))

df_silver_rev=(df_bronze_rev
    .withColumn("id_filme",F.trim(F.col("id").cast("string")))
    .withColumn("nome_usuario",F.trim(F.col("nome").cast("string")))
    .withColumn("nota_txt",F.regexp_replace(F.trim(F.col("nota").cast("string")),",","."))
    .withColumn("nota_usuario",numero_seguro("nota_txt","double"))
    .withColumn("nota_usuario",F.when(F.col("nota_usuario").between(0,10),F.col("nota_usuario")))
    .withColumn("comentario_usuario",
        F.when(F.col("comentario").isNull()|(F.trim(F.col("comentario").cast("string"))==""),"Sem comentário")
         .otherwise(F.trim(F.col("comentario").cast("string"))))
    .filter(F.col("id_filme").isNotNull())
    .select("id_filme","nome_usuario","nota_usuario","comentario_usuario")
    .dropDuplicates(["id_filme","nome_usuario","nota_usuario","comentario_usuario"]))

dq_check("silver.tb_avaliacoes_usuarios","nota 0-10 ou NULL",df_silver_rev,
         F.col("nota_usuario").isNull()|F.col("nota_usuario").between(0,10))
dq_check_unique("silver.tb_avaliacoes_usuarios","avaliacao única",df_silver_rev,
                ["id_filme","nome_usuario","nota_usuario","comentario_usuario"])

salvar_silver(df_silver_rev,"tb_avaliacoes_usuarios")
display(df_silver_rev.limit(10))

[PASS] silver.tb_avaliacoes_usuarios | nota 0-10 ou NULL | 0/32412 linhas falharam
[PASS] silver.tb_avaliacoes_usuarios | avaliacao única | 0 chaves duplicadas de 32412 linhas
Gravado: medalhao_cinema.silver.tb_avaliacoes_usuarios


id_filme,nome_usuario,nota_usuario,comentario_usuario
637007,Lucas Reis 602,3.9,Sem comentário
1100094,Gabriel Carvalho 581,6.2,"Aceitável, mas esperava mais."
628575,Alexandre Barbosa 220,0.3,Péssimo em todos os sentidos.
573249,Rodrigo Oliveira 273,0.5,Péssimo em todos os sentidos.
592539,Pedro Costa 181,4.8,"Não gostei, história confusa."
464493,Adriana Dias 257,0.4,Péssimo em todos os sentidos.
1199748,Eduardo Dias 177,6.0,"Poderia ser melhor, mas não é ruim."
640543,Cristina Monteiro 310,6.4,Sem comentário
599134,Larissa Lopes 330,2.6,Péssimo em todos os sentidos.
424011,Vinícius Ferreira 405,0.5,Não recomendo de jeito nenhum.


## silver.tb_generos

In [0]:
# troco ; por , e faco split + explode: 1 linha por filme x genero
# limpo o lixo do column shift: vazio, numero e texto descritivo (genero tem no maximo 3 palavras e so letras)
df_bronze_cred=mais_recente(
    spark.table(table(bronze_schema,"tb_credits_and_tags")).withColumn("id_filme",F.trim(F.col("id").cast("string"))),
    "id_filme")

df_silver_generos=(df_bronze_cred
    .filter(F.col("id_filme").isNotNull())
    .withColumn("nome_genero",F.explode(F.split(F.regexp_replace(F.col("genres").cast("string"),";",","),",")))
    .withColumn("nome_genero",F.initcap(F.trim(F.col("nome_genero"))))
    .filter(F.col("nome_genero").rlike("^[A-Za-zÀ-ÿ&' -]{2,30}$"))
    .filter(F.size(F.split(F.col("nome_genero")," "))<=3)
    .select("id_filme","nome_genero")
    .dropDuplicates(["id_filme","nome_genero"]))

dq_check_unique("silver.tb_generos","filme x genero único",df_silver_generos,["id_filme","nome_genero"])

salvar_silver(df_silver_generos,"tb_generos")
display(df_silver_generos.groupBy("nome_genero").count().orderBy(F.col("count").desc()))

[PASS] silver.tb_generos | filme x genero único | 0 chaves duplicadas de 132718 linhas
Gravado: medalhao_cinema.silver.tb_generos


nome_genero,count
Drama,30760
Documentary,18847
Comedy,17518
Thriller,9519
Horror,9257
Romance,7047
Action,5570
Crime,4349
Animation,4197
Tv Movie,3703


## silver.tb_pessoas_empresas

In [0]:
# junto as 4 colunas numa tabela so: cast=Ator, directors=Diretor, writers=Roteirista, production_companies=Produtora
# split por , ou ; + explode, capitalizacao padrao (initcap), tiro vazio/numero/texto longo e duplicados
def explodir(coluna,tipo):
    return (df_bronze_cred
        .select("id_filme",F.explode(F.split(F.col(coluna).cast("string"),"[,;]")).alias("nome_entidade"))
        .withColumn("nome_entidade",F.initcap(F.trim(F.regexp_replace(F.col("nome_entidade"),"\\s+"," "))))
        .withColumn("tipo_entidade",F.lit(tipo)))

df_silver_pessoas=(explodir("cast","Ator")
    .unionByName(explodir("directors","Diretor"))
    .unionByName(explodir("writers","Roteirista"))
    .unionByName(explodir("production_companies","Produtora"))
    .filter(F.col("id_filme").isNotNull())
    .filter(F.col("nome_entidade").isNotNull()&(F.col("nome_entidade")!=""))
    .filter(~F.col("nome_entidade").rlike("^[0-9.,\\s-]+$"))
    .filter((F.length("nome_entidade")<=60)&(F.size(F.split(F.col("nome_entidade")," "))<=6))
    .select("id_filme","nome_entidade","tipo_entidade")
    .dropDuplicates(["id_filme","nome_entidade","tipo_entidade"]))

dq_check("silver.tb_pessoas_empresas","tipo válido",df_silver_pessoas,
         F.col("tipo_entidade").isin("Ator","Diretor","Roteirista","Produtora"))
dq_check_unique("silver.tb_pessoas_empresas","sem duplicado",df_silver_pessoas,["id_filme","nome_entidade","tipo_entidade"])

salvar_silver(df_silver_pessoas,"tb_pessoas_empresas")
display(df_silver_pessoas.groupBy("tipo_entidade").count())

[PASS] silver.tb_pessoas_empresas | tipo válido | 0/906012 linhas falharam
[PASS] silver.tb_pessoas_empresas | sem duplicado | 0 chaves duplicadas de 906012 linhas
Gravado: medalhao_cinema.silver.tb_pessoas_empresas


tipo_entidade,count
Roteirista,137167
Diretor,107801
Ator,541925
Produtora,119119


In [0]:
# conferencia: tem que aparecer as 7 tabelas aqui antes de rodar a gold
display(spark.sql(f"SHOW TABLES IN {silver_schema}"))

database,tableName,isTemporary
silver,filmes,false
silver,tb_avaliacoes_usuarios,false
silver,tb_cotacao_dolar,false
silver,tb_financeiro_filmes,false
silver,tb_generos,false
silver,tb_info_filmes,false
silver,tb_metricas_engajamento,false
silver,tb_pessoas_empresas,false
